# Amazon Bedrock AgentCore Runtime上でのMCPサーバーのホスティング - OAuthインバウンド認証

## 概要

このチュートリアルでは、Amazon Bedrock AgentCore Runtime上でMCP（Model Context Protocol）サーバーをホスティングする方法を学習します。Amazon Bedrock AgentCore Python SDKを使用して、MCPツールをAmazon Bedrock AgentCoreと互換性のあるMCPサーバーとしてラップします。

Amazon Bedrock AgentCore Python SDKはMCPサーバーの実装詳細を処理するため、ツールのコア機能に集中できます。コードをAgentCore標準化されたMCPプロトコルコントラクトに変換して、直接通信を可能にします。

### チュートリアルの詳細

| 情報               | 詳細                                                       |
|:-------------------|:-----------------------------------------------------------|
| チュートリアルタイプ | ツールのホスティング                                       |
| ツールタイプ       | MCPサーバー                                                |
| チュートリアル構成要素 | AgentCore Runtime上でのMCPサーバーのホスティング           |
| チュートリアル垂直領域 | クロス垂直領域                                             |
| 例の複雑さ         | 簡単                                                       |
| 使用SDK            | Amazon BedrockAgentCore Python SDKおよびMCP               |

### チュートリアルアーキテクチャ

このチュートリアルでは、MCPサーバーをAgentCore runtimeにデプロイする方法について説明します。

デモンストレーションの目的で、3つのツールを持つシンプルなMCPサーバーを使用します：`add_numbers`、`multiply_numbers`、`greet_user`

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### チュートリアルの主な機能

* カスタムツールを使用したMCPサーバーの作成
* MCPサーバーのローカルテスト
* Amazon Bedrock AgentCore Runtime上でのMCPサーバーのホスティング
* 認証を使用したデプロイ済みMCPサーバーの呼び出し


## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Python 3.10+
* AWS認証情報の設定
* Amazon Bedrock AgentCore SDK
* MCP（Model Context Protocol）ライブラリ
* Dockerの実行

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [4]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os
import sys

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import setup_cognito_user_pool

sys.path[0]: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials


In [7]:
boto_session = Session()
region = boto_session.region_name

ssm_client = boto_session.client('ssm', region_name=region)
secrets_client = boto_session.client('secretsmanager', region_name=region)
agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)

tool_name = "mcp_server_agentcore"

## MCP（Model Context Protocol）の理解

MCPは、AIモデルが外部データとツールに安全にアクセスできるようにするプロトコルです。主要な概念：

* **ツール**: AIがアクションを実行するために呼び出すことができる関数
* **Streamable HTTP**: AgentCore Runtimeで使用されるトランスポートプロトコル
* **セッション分離**: 各クライアントは`Mcp-Session-Id`ヘッダーを介して分離されたセッションを取得
* **ステートレス操作**: サーバーはスケーラビリティのためにステートレス操作をサポートする必要があります

AgentCore Runtimeは、MCPサーバーがデフォルトパスとして`0.0.0.0:8000/mcp`でホスティングされることを期待します。

### プロジェクト構造

適切な構造でプロジェクトを設定しましょう：

```
mcp_server_project/
├── mcp_server.py              # メインMCPサーバーコード
├── my_mcp_client.py          # ローカルテストクライアント
├── my_mcp_client_remote.py   # リモートテストクライアント
├── requirements.txt          # 依存関係
└── __init__.py              # Pythonパッケージマーカー
```

## MCPサーバーの作成

3つのシンプルなツールを使用してMCPサーバーを作成しましょう。サーバーは`stateless_http=True`を使用したFastMCPを使用します。これはAgentCore Runtimeとの互換性に必要です。

In [3]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """2つの数値を足し合わせる"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """2つの数値を掛け合わせる"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """名前でユーザーを挨拶する"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Writing mcp_server.py


### このコードの動作

* **FastMCP**: ツールをホスティングできるMCPサーバーを作成
* **@mcp.tool()**: Python関数をMCPツールに変換するデコレータ
* **stateless_http=True**: AgentCore Runtimeとの互換性に必要
* **ツール**: 異なるタイプの操作を示す3つのシンプルなツール

## ローカルテストクライアントの作成

AgentCore Runtimeにデプロイする前に、MCPサーバーをローカルでテストするクライアントを作成しましょう：

In [4]:
%%writefile my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

Writing my_mcp_client.py


### ローカルでのテスト

MCPサーバーをローカルでテストするには：

1. **ターミナル1**: MCPサーバーを起動
   ```bash
   python mcp_server.py
   ```
   
2. **ターミナル2**: テストクライアントを実行
   ```bash
   python my_mcp_client.py
   ```

出力に3つのツールがリストされているはずです。

## 認証のためのAmazon Cognitoの設定

AgentCore Runtimeには認証が必要です。デプロイされたMCPサーバーにアクセスするためのJWTトークンを提供するために、Amazon Cognitoを使用します。

In [5]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

Setting up Amazon Cognito user pool...
Pool id: ap-northeast-1_P2JZfSabH
Discovery URL: https://cognito-idp.ap-northeast-1.amazonaws.com/ap-northeast-1_P2JZfSabH/.well-known/openid-configuration
Client ID: 72697u202lc9r39evtlu6l3fgh
Bearer Token: eyJraWQiOiJHbEt4K2NlWnJ4ZGxDK2lIXC9yUlZvd1liNlJhK1NqRStCc2FVbjRcL0psUTg9IiwiYWxnIjoiUlMyNTYifQ.eyJzdWIiOiIyN2I0N2EyOC1iMDAxLTcwYmQtZjFmMy01MGY4NzAxOWMyMWUiLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAuYXAtbm9ydGhlYXN0LTEuYW1hem9uYXdzLmNvbVwvYXAtbm9ydGhlYXN0LTFfUDJKWmZTYWJIIiwiY2xpZW50X2lkIjoiNzI2OTd1MjAybGM5cjM5ZXZ0bHU2bDNmZ2giLCJvcmlnaW5fanRpIjoiZTlmZTNmZWItNWFkNy00YzhmLTg3MmItMjc5MjRjNmNiNTJjIiwiZXZlbnRfaWQiOiJhMGEzOTRlYy01ZTdiLTRlNGMtYWRkYi1lZjAwYmU2MDZhNzIiLCJ0b2tlbl91c2UiOiJhY2Nlc3MiLCJzY29wZSI6ImF3cy5jb2duaXRvLnNpZ25pbi51c2VyLmFkbWluIiwiYXV0aF90aW1lIjoxNzY3NTAyNzM2LCJleHAiOjE3Njc1MDYzMzYsImlhdCI6MTc2NzUwMjczNiwianRpIjoiMDEwMDEyZmUtNDNhMC00YTkwLWE1ZGQtNjgzNWJkM2VkNTcxIiwidXNlcm5hbWUiOiJ0ZXN0dXNlciJ9.yZSblQmMcu4BXMJFPWDutIXHdz_OysEx9j7nB-F0cYapN

## AgentCore Runtimeデプロイメントの設定

次に、スターターキットを使用して、エントリーポイント、作成した実行ロール、およびrequirementsファイルを使用してAgentCore Runtimeデプロイメントを設定します。また、起動時にAmazon ECRリポジトリを自動作成するようにスターターキットを設定します。

設定ステップ中に、アプリケーションコードに基づいてDockerファイルが生成されます。

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [8]:
print(f"Using AWS region: {region}")

required_files = ['mcp_server.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            cognito_config['client_id']
        ],
        "discoveryUrl": cognito_config['discovery_url'],
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name=tool_name
)
print("Configuration completed ✓")

Entrypoint parsed: file=/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/mcp_server.py, bedrock_agentcore_name=mcp_server
Configuring BedrockAgentCore agent: mcp_server_agentcore


Using AWS region: ap-northeast-1
All required files found ✓
Configuring AgentCore Runtime...


Generated .dockerignore
Generated Dockerfile: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/Dockerfile
Generated .dockerignore: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/.dockerignore
Setting 'mcp_server_agentcore' as default agent
Bedrock AgentCore configured: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/.bedrock_agentcore.yaml


Configuration completed ✓


## AgentCore RuntimeへのMCPサーバーの起動

Dockerファイルができたので、MCPサーバーをAgentCore Runtimeに起動しましょう。これにより、Amazon ECRリポジトリとAgentCore Runtimeが作成されます。

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [9]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'mcp_server_agentcore' to account 195049633937 (ap-northeast-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: mcp_server_agentcore


Launching MCP server to AgentCore Runtime...
This may take several minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-mcp_server_agentcore


✅ ECR repository available: 195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-mcp_server_agentcore
Getting or creating execution role for agent: mcp_server_agentcore
Using AWS region: ap-northeast-1, account ID: 195049633937
Role name: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-7f3ae149b4
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-7f3ae149b4
Starting execution role creation process for agent: mcp_server_agentcore
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-7f3ae149b4
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-7f3ae149b4
✓ Role created: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-7f3ae149b4
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-mcp_server_agentcore
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKRuntime-a

Launch completed ✓
Agent ARN: arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/mcp_server_agentcore-so3UlE87aL
Agent ID: mcp_server_agentcore-so3UlE87aL


## リモートアクセス用の設定の保存

デプロイされたMCPサーバーを呼び出す前に、Agent ARNとCognito設定をAWS Systems Manager Parameter StoreとAWS Secrets Managerに保存して、簡単に取得できるようにしましょう：

In [10]:
import boto3
import json

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

✓ Cognito credentials stored in Secrets Manager
✓ Agent ARN stored in Parameter Store

Configuration stored successfully!
Agent ARN: arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/mcp_server_agentcore-so3UlE87aL


## リモートテストクライアントの作成

デプロイされたMCPサーバーをテストするクライアントを作成しましょう。このクライアントはAWSから必要な認証情報を取得し、デプロイされたサーバーに接続します：

In [11]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
import base64
import time
from boto3.session import Session
from datetime import timedelta
import traceback

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """Refresh access token using refresh token"""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """Check token expiry and refresh if needed"""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except Exception as e:
        print("🔄 Invalid token, refreshing...", e)
        traceback.print_exc()
        return get_refresh_token(client_id, refresh_token, region)

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        # Validate and refresh token if needed
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

Writing my_mcp_client_remote.py


## デプロイされたMCPサーバーのテスト

リモートクライアントを使用してデプロイされたMCPサーバーをテストしましょう：

In [12]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

Testing deployed MCP server...
Using AWS region: ap-northeast-1
Retrieved Agent ARN: arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/mcp_server_agentcore-so3UlE87aL
✓ Retrieved credentials from Secrets Manager

Connecting to: https://bedrock-agentcore.ap-northeast-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aap-northeast-1%3A195049633937%3Aruntime%2Fmcp_server_agentcore-so3UlE87aL/invocations?qualifier=DEFAULT
Headers configured ✓

🔄 Initializing MCP session...
✓ MCP session initialized

🔄 Listing available tools...

📋 Available MCP Tools:
🔧 add_numbers
   Description: 2つの数値を足し合わせる
   Parameters: ['a', 'b']

🔧 multiply_numbers
   Description: 2つの数値を掛け合わせる
   Parameters: ['a', 'b']

🔧 greet_user
   Description: 名前でユーザーを挨拶する
   Parameters: ['name']

✅ Successfully connected to MCP server!
Found 3 tools available.


## リモートでのMCPツールの呼び出し

ツールをリストするだけでなく、呼び出してMCP機能全体を実証する拡張クライアントを作成しましょう：

In [13]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
import base64
import time
from boto3.session import Session
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """Refresh access token using refresh token"""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """Check token expiry and refresh if needed"""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except:
        print("🔄 Invalid token, refreshing...")
        return get_refresh_token(client_id, refresh_token, region)

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        # Validate and refresh token if needed
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testing add_numbers(5, 3)...")
                    add_result = await session.call_tool(
                        name="add_numbers",
                        arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                try:
                    print("\n✖️  Testing multiply_numbers(4, 7)...")
                    multiply_result = await session.call_tool(
                        name="multiply_numbers",
                        arguments={"a": 4, "b": 7}
                    )
                    print(f"   Result: {multiply_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                try:
                    print("\n👋 Testing greet_user('Alice')...")
                    greet_result = await session.call_tool(
                        name="greet_user",
                        arguments={"name": "Alice"}
                    )
                    print(f"   Result: {greet_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                print("\n✅ MCP tool testing completed!")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

Writing invoke_mcp_tools.py


## ツール呼び出しのテスト

実際にMCPツールを呼び出してテストしましょう：

In [14]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

Testing MCP tool invocation...
Using AWS region: ap-northeast-1
Retrieved Agent ARN: arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/mcp_server_agentcore-so3UlE87aL
✓ Retrieved credentials from Secrets Manager

Connecting to: https://bedrock-agentcore.ap-northeast-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aap-northeast-1%3A195049633937%3Aruntime%2Fmcp_server_agentcore-so3UlE87aL/invocations?qualifier=DEFAULT

🔄 Initializing MCP session...
✓ MCP session initialized

🔄 Listing available tools...

📋 Available MCP Tools:
🔧 add_numbers: 2つの数値を足し合わせる
🔧 multiply_numbers: 2つの数値を掛け合わせる
🔧 greet_user: 名前でユーザーを挨拶する

🧪 Testing MCP Tools:

➕ Testing add_numbers(5, 3)...
   Result: 8

✖️  Testing multiply_numbers(4, 7)...
   Result: 28

👋 Testing greet_user('Alice')...
   Result: Hello, Alice! Nice to meet you.

✅ MCP tool testing completed!


## 次のステップ

AgentCore RuntimeにMCPサーバーを正常にデプロイしたので、以下を実行できます：

1. **ツールの追加**: 追加のツールでMCPサーバーを拡張
2. **カスタム認証**: カスタムJWTオーソライザーを実装
3. **統合**: 他のAgentCoreサービスと統合

## クリーンアップ（オプション）

このチュートリアル中に作成されたリソースをクリーンアップする場合は、次のセルを実行してください：

In [ ]:
# print("🗑️  Starting cleanup process...")

# try:
#     ssm_client.delete_parameter(Name='/mcp_server/runtime/agent_arn')
#     print("✓ Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("ℹ️  Parameter Store parameter not found")

# try:
#     secrets_client.delete_secret(
#         SecretId='mcp_server/cognito/credentials',
#         ForceDeleteWithoutRecovery=True
#     )
#     print("✓ Secrets Manager secret deleted")
# except secrets_client.exceptions.ResourceNotFoundException:
#     print("ℹ️  Secrets Manager secret not found")

# print("\n✅ Cleanup completed successfully!")

In [ ]:
# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True
# )

# 🎉 おめでとうございます！

以下を正常に完了しました：

✅ **カスタムツールを使用してMCPサーバーを作成**  
✅ **MCPクライアントを使用してローカルでテスト**  
✅ **Amazon Cognitoで認証を設定**  
✅ **AgentCore Runtimeを使用してAWSにデプロイ**  
✅ **適切な認証でリモートから呼び出し**  
✅ **MCPの概念とベストプラクティスを学習**  

MCPサーバーは現在、Amazon Bedrock AgentCore Runtime上で実行されており、本番環境での使用準備が整っています！

## まとめ

このチュートリアルでは、以下を学習しました：
- FastMCPを使用したMCPサーバーの構築
- AgentCore互換性のためのステートレスHTTPトランスポートの設定
- Amazon Cognitoを使用したJWT認証の設定
- AWS上でのMCPサーバーのデプロイと管理
- ローカルとリモートの両方でのテスト
- ツール呼び出しのためのMCPクライアントの使用

デプロイされたMCPサーバーは、より大きなAIアプリケーションとワークフローに統合できるようになりました！